In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 0ba5f038-b770-4bb4-8346-bfbcd2d6884f, 4, Finished, Available, Finished, False)

###### <mark>**STEP 1 — Read Bronze as Stream**</mark>
👉 **<mark>Important:</mark>**

**We are now streaming from Delta table (not files)
This is stream-on-stream (Delta streaming)**


In [3]:
silver_input_df = spark.readStream \
    .table("bronze_order_events_stream")

StatementMeta(, 0ba5f038-b770-4bb4-8346-bfbcd2d6884f, 5, Finished, Available, Finished, False)

###### <mark>**STEP 2 — Deduplication**</mark>

In [4]:
dedup_df = silver_input_df.dropDuplicates(["order_id"])

StatementMeta(, 0ba5f038-b770-4bb4-8346-bfbcd2d6884f, 6, Finished, Available, Finished, False)

###### <mark>**STEP 3 — Add Watermark**</mark>

In [5]:
from pyspark.sql.functions import col, to_timestamp

clean_df = dedup_df \
    .withColumn("event_time", to_timestamp(col("event_time"))) \
    .withWatermark("event_time", "10 minutes")

StatementMeta(, 0ba5f038-b770-4bb4-8346-bfbcd2d6884f, 7, Finished, Available, Finished, False)

###### <mark>**STEP 4 — Business Transformation**</mark>

In [6]:
from pyspark.sql.functions import when

final_df = clean_df.withColumn(
    "order_status",
    when(col("event_type") == "order_placed", "CREATED")
    .when(col("event_type") == "payment_success", "PAID")
    .otherwise("UNKNOWN")
)

StatementMeta(, 0ba5f038-b770-4bb4-8346-bfbcd2d6884f, 8, Finished, Available, Finished, False)

###### <mark>**STEP 5 — Write Silver Stream**</mark>

In [7]:
silver_query = final_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "Files/streaming/checkpoints/silver_order_events") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .toTable("silver_order_events_stream")

StatementMeta(, 0ba5f038-b770-4bb4-8346-bfbcd2d6884f, 9, Finished, Available, Finished, False)

###### <mark>**STEP 6 — Validate Output**</mark>

In [8]:
spark.sql("""
SELECT * 
FROM silver_order_events_stream
ORDER BY order_id
""").show(truncate=False)

StatementMeta(, 0ba5f038-b770-4bb4-8346-bfbcd2d6884f, 10, Finished, Available, Finished, False)

+--------+-----------+----------+---------------+-------------------+------+-------+------------+
|order_id|customer_id|product_id|event_type     |event_time         |amount|status |order_status|
+--------+-----------+----------+---------------+-------------------+------+-------+------------+
|900001  |101        |2001      |order_placed   |2026-03-31 10:00:00|120.5 |created|CREATED     |
|900002  |102        |2002      |order_placed   |2026-03-31 10:01:00|80.0  |created|CREATED     |
|900003  |103        |2003      |payment_success|2026-03-31 10:02:00|220.0 |paid   |PAID        |
|900004  |104        |2004      |order_placed   |2026-03-31 10:03:00|55.75 |created|CREATED     |
|900005  |105        |2005      |order_placed   |2026-03-31 10:05:00|310.25|created|CREATED     |
|900006  |106        |2006      |payment_success|2026-03-31 10:06:00|99.99 |paid   |PAID        |
|900007  |107        |2007      |order_placed   |2026-03-31 10:07:00|145.1 |created|CREATED     |
+--------+----------